# Lex Triage — Interactive HITL Pipeline Runner

**Purpose:** Run the full legal-triage LangGraph pipeline over a batch of emails with live
Human-in-the-Loop (HITL) review for PI leads.  

**Two execution modes:**
- **Auto mode** *(default — great for development / CI)*: PI leads are automatically
  approved/rejected based on appraisal score + configurable probability. No human
  interaction required.
- **Interactive HITL mode**: The pipeline pauses on every PI lead and shows an
  Approve / Reject / Reclassify review panel. Click a button to continue.

**After the run** all statistics are rendered inline: routing distribution, confusion
matrix, latency histogram, cost breakdown, and an executive CEO summary.

---
**Quick start (fresh kernel):**
1. `Run All Cells` — this installs deps, compiles the graph, loads emails
2. Adjust the configuration sliders / toggles if needed
3. Click **▶ Run Pipeline**
4. For interactive HITL, approve/reject emails as review panels appear
5. Scroll down for statistics once processing completes

In [1]:
# Install / upgrade required packages (safe to re-run)
# %pip install -q ipywidgets plotly pandas scikit-learn nest_asyncio

In [2]:
"""Kernel bootstrap — sys.path, env vars."""
import os
import sys
from pathlib import Path

# Locate repo root regardless of whether notebook is opened from repo root
# or from the notebooks/ sub-directory.
REPO_ROOT = Path.cwd()
for _ in range(3):
    if (REPO_ROOT / "apps").exists():
        break
    REPO_ROOT = REPO_ROOT.parent
else:
    raise RuntimeError(
        f"Cannot locate repo root from {Path.cwd()!r}. "
        "Run this notebook from the repo root or the notebooks/ directory."
    )

LEGAL_TRIAGE_SRC = REPO_ROOT / "apps" / "legal-triage" / "src"
if str(LEGAL_TRIAGE_SRC) not in sys.path:
    sys.path.insert(0, str(LEGAL_TRIAGE_SRC))

# Default to offline tier3 stubs so the notebook works without API keys.
# Switch to tier1 / tier2 for real model calls.
os.environ.setdefault("LLM_TIER", "tier3")
os.environ.setdefault("LANGSMITH_TRACING", "false")

print(f"Repo root    : {REPO_ROOT}")
print(f"LLM_TIER     : {os.environ['LLM_TIER']}")
print(f"legal-triage : {LEGAL_TRIAGE_SRC}")

Repo root    : /Users/wojciechkrukar/code/lex-triage-agent
LLM_TIER     : tier1             # tier1|tier2|tier3
legal-triage : /Users/wojciechkrukar/code/lex-triage-agent/apps/legal-triage/src


In [3]:
"""Imports — legal_triage and notebook stack."""
import asyncio
import json
import random
import traceback
from collections import Counter
from typing import Any

import nest_asyncio
nest_asyncio.apply()  # allow asyncio.run() inside running Jupyter event loop

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from legal_triage.graph import get_compiled_graph
from legal_triage.state import initial_state
from legal_triage.nodes.router import router_node
import legal_triage.hitl_queue as hitl_queue

GRAPH = get_compiled_graph()
print("✅ Pipeline graph compiled successfully")

✅ Pipeline graph compiled successfully


In [4]:
"""Load labelled emails from the dataset-generator output directory."""

# Ground-truth field prefixes — stripped before the pipeline sees the email
_GT_PREFIXES = ("gt_",)

# Expected sink per GT class (mirrors eval.py / router.py)
_CLASS_TO_SINK = {
    "pi_lead":      "NewLead",
    "general_legal": "GeneralLegal",
    "spam":         "Refused-Spam",
    "invoice":      "Refused-Invoice",
    "other":        "Refused-Other",
}


def _strip_gt(record: dict) -> dict:
    """Return a copy without any ground-truth fields."""
    return {k: v for k, v in record.items()
            if not any(k.startswith(p) for p in _GT_PREFIXES)}


def load_emails(max_emails: int = 50) -> tuple[list[dict], list[str]]:
    """
    Load emails from the nearest available dataset file.

    Returns:
        (public_records, gt_classes)  —  parallel lists.
        public_records : GT fields stripped (safe to pass to pipeline).
        gt_classes     : ground-truth class label for confusion matrix.
    """
    candidates = [
        REPO_ROOT / "apps" / "dataset-generator" / "out" / "emails_gt.jsonl",
        REPO_ROOT / "apps" / "dataset-generator" / "out" / "emails_demo_gt.jsonl",
        REPO_ROOT / "apps" / "dataset-generator" / "out" / "emails.jsonl",
        REPO_ROOT / "apps" / "dataset-generator" / "out" / "emails_demo.jsonl",
    ]
    for path in candidates:
        if path.exists():
            raw: list[dict] = []
            with path.open() as fh:
                for line in fh:
                    if line.strip():
                        raw.append(json.loads(line))
                    if len(raw) >= max_emails:
                        break
            print(f"📂 Loaded {len(raw)} emails from {path.name}")
            gt_classes = [r.get("gt_class", "other") for r in raw]
            public = [_strip_gt(r) for r in raw]
            return public, gt_classes

    print("⚠️  No dataset file found — falling back to 20 synthetic emails")
    return _synthetic_fallback(min(max_emails, 20))


def _synthetic_fallback(n: int) -> tuple[list[dict], list[str]]:
    templates = (
        [{"email_id": f"syn-pi-{i:03d}",
          "subject": "Car accident — need attorney",
          "body": "I was rear-ended at a red light and suffered whiplash. "
                  "The other driver was clearly at fault. Please help.",
          "sender": "victim@example.com",
          "attachments": []}
         for i in range(n // 2)]
        + [{"email_id": f"syn-gen-{i:03d}",
            "subject": "Contract review request",
            "body": "Could you review my lease agreement before I sign?",
            "sender": "client@example.com",
            "attachments": []}
           for i in range(n - n // 2)]
    )
    gt_classes = (["pi_lead"] * (n // 2) + ["general_legal"] * (n - n // 2))
    return templates[:n], gt_classes[:n]


# --- Eager load with default batch size ---------------------------------
MAX_EMAILS = 30   # change before running — or use the slider below

ALL_EMAILS, ALL_GT = load_emails(max_emails=500)   # load pool
print(f"Pool size: {len(ALL_EMAILS)}  |  GT classes: {Counter(ALL_GT)}")

📂 Loaded 120 emails from emails_gt.jsonl
Pool size: 120  |  GT classes: Counter({'pi_lead': 80, 'spam': 15, 'invoice': 10, 'general_legal': 10, 'other': 5})


In [5]:
"""HitlBatchRunner — processes emails, shows HITL widgets, accumulates results."""

class HitlBatchRunner:
    """
    Two-stage pipeline runner with optional interactive HITL UI.

    Stage 1: graph.invoke(state)  →  if hitl_required and no human_decision:
    Stage 2: collect decision (widget or auto) → call router_node directly.

    Auto mode heuristic:
      appraisal_score >= 0.75  → approve
      appraisal_score <  0.35  → reject
      else                     → approve with probability `auto_approve_rate`
    """

    def __init__(
        self,
        emails: list[dict],
        gt_classes: list[str],
        graph,
        *,
        auto_mode: bool = True,
        auto_approve_rate: float = 0.9,
    ):
        self.emails = emails
        self.gt_classes = gt_classes
        self.graph = graph
        self.auto_mode = auto_mode
        self.auto_approve_rate = auto_approve_rate

        self.results: list[dict] = []
        self._progress_bar: widgets.IntProgress | None = None
        self._status_label: widgets.Label | None = None
        self._hitl_output: widgets.Output | None = None

    # ------------------------------------------------------------------
    # Public entry point
    # ------------------------------------------------------------------

    def run(self) -> list[dict]:
        """Process all emails and return a list of result dicts."""
        hitl_queue.reset()
        self.results = []
        total = len(self.emails)

        # Build progress UI
        self._progress_bar = widgets.IntProgress(
            value=0, min=0, max=total,
            description="Progress:",
            bar_style="info",
            layout=widgets.Layout(width="60%")
        )
        self._status_label = widgets.Label(value="Starting…")
        self._hitl_output = widgets.Output()

        display(widgets.VBox([
            widgets.HBox([self._progress_bar, self._status_label]),
            self._hitl_output,
        ]))

        for i, (record, gt_class) in enumerate(zip(self.emails, self.gt_classes)):
            self._status_label.value = (
                f"[{i+1}/{total}] Processing email {record.get('email_id','?')[:24]}…"
                f"  |  {hitl_queue.status_message()}"
            )

            state = initial_state(
                email_id=record.get("email_id", f"nb-{i:05d}"),
                raw_email=record.get("body", record.get("raw_email", "")),
                attachments=record.get("attachments", []),
            )

            result_state = self._run_single(state, i + 1, total, record)
            self.results.append({
                "email_id":      result_state.get("email_id"),
                "gt_class":      gt_class,
                "predicted_class": result_state.get("email_class"),
                "terminal_sink": result_state.get("terminal_sink"),
                "expected_sink": _CLASS_TO_SINK.get(gt_class, "Refused-Other"),
                "hitl_required": result_state.get("hitl_required", False),
                "human_decision": result_state.get("human_decision"),
                "appraisal_score": result_state.get("appraisal_score"),
                "class_confidence": result_state.get("class_confidence"),
                "total_cost_usd":  result_state.get("total_cost_usd", 0.0),
                "total_latency_ms": result_state.get("total_latency_ms", 0),
                "errors":        result_state.get("errors", []),
            })

            self._progress_bar.value = i + 1
            if i + 1 == total:
                self._progress_bar.bar_style = "success"

        self._status_label.value = (
            f"✅ Done — {total} emails processed  |  "
            f"HITL reviews: {sum(1 for r in self.results if r['hitl_required'])}"
        )
        return self.results

    # ------------------------------------------------------------------
    # Single-email logic (two-stage HITL)
    # ------------------------------------------------------------------

    def _run_single(self, state, email_num: int, total: int, record: dict) -> dict:
        try:
            result = dict(self.graph.invoke(state))
        except Exception as exc:
            result = dict(state)
            result["errors"] = result.get("errors", []) + [str(exc)]
            result["terminal_sink"] = None
            return result

        # Stage 2: handle HITL if triggered
        if result.get("hitl_required") and result.get("human_decision") is None:
            if self.auto_mode:
                decision = self._auto_decide(result)
            else:
                decision = self._interactive_decide(result, email_num, total, record)

            result["human_decision"] = decision
            try:
                router_result = router_node(result)
                result.update(router_result)
            except Exception as exc:
                result["errors"] = result.get("errors", []) + [f"router: {exc}"]

            hitl_queue.dequeue()

        return result

    # ------------------------------------------------------------------
    # Auto-approve heuristic
    # ------------------------------------------------------------------

    def _auto_decide(self, state: dict) -> str:
        score = state.get("appraisal_score") or 0.5
        if score >= 0.75:
            return "approve"
        if score < 0.35:
            return "reject"
        return "approve" if random.random() < self.auto_approve_rate else "reject"

    # ------------------------------------------------------------------
    # Interactive HITL widget (asyncio-based, works in Jupyter)
    # ------------------------------------------------------------------

    def _interactive_decide(
        self, state: dict, email_num: int, total: int, record: dict
    ) -> str:
        loop = asyncio.get_event_loop()
        return loop.run_until_complete(
            self._show_review_widget(state, email_num, total, record)
        )

    async def _show_review_widget(
        self, state: dict, email_num: int, total: int, record: dict
    ) -> str:
        """Render a review card with 3 buttons; await user click."""
        decision_future: asyncio.Future[str] = asyncio.get_event_loop().create_future()

        # --- pull display fields -----------------------------------------
        subject    = record.get("subject", "(no subject)")[:120]
        sender     = record.get("sender", "unknown")
        body_snip  = (record.get("body", "") or "")[:600]
        cls        = state.get("email_class") or "unknown"
        conf       = state.get("class_confidence") or 0.0
        appraisal  = (state.get("appraisal_draft") or "")[:500]
        score      = state.get("appraisal_score") or 0.0
        queue_d    = hitl_queue.depth()

        # --- review card HTML --------------------------------------------
        review_html = HTML(f"""
        <div style="border:2px solid #2E86AB;border-radius:8px;padding:0;
                    margin:12px 0;font-family:system-ui,sans-serif;max-width:800px">
          <div style="background:#2E86AB;color:white;padding:10px 16px;
                      border-radius:6px 6px 0 0;display:flex;justify-content:space-between">
            <b>&#128269; HITL REVIEW &nbsp; #{email_num} / {total}</b>
            <span style="font-size:0.85em">Queue depth: {queue_d}</span>
          </div>
          <div style="padding:14px 16px">
            <table style="width:100%;border-collapse:collapse;margin-bottom:10px">
              <tr>
                <td style="width:80px;color:#666">Class</td>
                <td><b>{cls}</b></td>
                <td style="width:100px;color:#666">Confidence</td>
                <td><b>{conf:.0%}</b></td>
                <td style="width:120px;color:#666">Appraisal score</td>
                <td><b>{score:.2f}</b></td>
              </tr>
              <tr>
                <td style="color:#666">From</td>
                <td colspan="5">{sender}</td>
              </tr>
              <tr>
                <td style="color:#666">Subject</td>
                <td colspan="5"><b>{subject}</b></td>
              </tr>
            </table>
            <details open>
              <summary style="cursor:pointer;font-weight:bold">Email body</summary>
              <pre style="background:#f8f8f8;padding:10px;border-radius:4px;
                         white-space:pre-wrap;max-height:200px;overflow-y:auto;
                         font-size:0.88em">{body_snip}</pre>
            </details>
            <details>
              <summary style="cursor:pointer;font-weight:bold;margin-top:8px">Appraisal draft</summary>
              <pre style="background:#f8f8f8;padding:10px;border-radius:4px;
                         white-space:pre-wrap;max-height:200px;overflow-y:auto;
                         font-size:0.88em">{appraisal or '(no appraisal generated)'}</pre>
            </details>
          </div>
        </div>
        """)

        btn_approve = widgets.Button(
            description="✅ Approve → NewLead",
            button_style="success",
            layout=widgets.Layout(width="200px", height="38px")
        )
        btn_reject = widgets.Button(
            description="❌ Reject → Refused",
            button_style="danger",
            layout=widgets.Layout(width="200px", height="38px")
        )
        btn_reclassify = widgets.Button(
            description="🔄 Reclassify → General",
            button_style="warning",
            layout=widgets.Layout(width="210px", height="38px")
        )

        def make_handler(value: str):
            def handler(b):
                if not decision_future.done():
                    for btn in (btn_approve, btn_reject, btn_reclassify):
                        btn.disabled = True
                    decision_future.set_result(value)
            return handler

        btn_approve.on_click(make_handler("approve"))
        btn_reject.on_click(make_handler("reject"))
        btn_reclassify.on_click(make_handler("reclassify"))

        with self._hitl_output:  # type: ignore[union-attr]
            clear_output(wait=True)
            display(review_html)
            display(widgets.HBox([btn_approve, btn_reject, btn_reclassify]))

        # Yield control to event loop while waiting — allows button events to fire
        while not decision_future.done():
            await asyncio.sleep(0.05)

        with self._hitl_output:  # type: ignore[union-attr]
            clear_output(wait=True)
            display(HTML(
                f"<div style='padding:8px;background:#e8f5e9;border-radius:4px'>"
                f"&#x2714; Email #{email_num}: decision = "
                f"<b>{decision_future.result()}</b></div>"
            ))

        return decision_future.result()


print("✅ HitlBatchRunner class loaded")

✅ HitlBatchRunner class loaded


In [6]:
"""Configuration panel — adjust before clicking Run."""

_TIER_OPTIONS = [
    "tier3  (stub — offline, no API keys needed)",
    "tier2  (dev — Claude Haiku / GPT-4o-mini)",
    "tier1  (prod — Claude Opus / GPT-4o)",
]

w_tier = widgets.Dropdown(
    options=_TIER_OPTIONS,
    value=_TIER_OPTIONS[0],
    description="LLM tier:",
    layout=widgets.Layout(width="420px"),
    style={"description_width": "120px"},
)
w_max_emails = widgets.IntSlider(
    value=min(30, len(ALL_EMAILS)),
    min=5, max=len(ALL_EMAILS), step=5,
    description="Batch size:",
    continuous_update=False,
    layout=widgets.Layout(width="420px"),
    style={"description_width": "120px"},
)
w_auto_mode = widgets.Checkbox(
    value=True,
    description="Auto-approve mode  (uncheck for live interactive HITL)",
    indent=False,
)
w_auto_rate = widgets.FloatSlider(
    value=0.9,
    min=0.0, max=1.0, step=0.05,
    description="Auto-approve %:",
    readout_format=".0%",
    continuous_update=False,
    layout=widgets.Layout(width="420px"),
    style={"description_width": "120px"},
)

config_box = widgets.VBox([
    widgets.HTML("<b>Pipeline configuration</b>"),
    w_tier,
    w_max_emails,
    widgets.HTML("<hr style='margin:6px 0'>"),
    w_auto_mode,
    w_auto_rate,
])
display(config_box)

In [7]:
"""Run the pipeline.  Click the button — or call runner.run() manually."""

run_button = widgets.Button(
    description="▶  Run Pipeline",
    button_style="primary",
    icon="play",
    layout=widgets.Layout(width="200px", height="44px"),
)
run_output = widgets.Output()
RESULTS: list[dict] = []   # populated after run
RUNNER: HitlBatchRunner | None = None

def on_run_clicked(b):
    global RESULTS, RUNNER
    with run_output:
        clear_output(wait=True)

        # Apply configuration
        tier = w_tier.value.split()[0]   # e.g. "tier3"
        os.environ["LLM_TIER"] = tier

        n = w_max_emails.value
        emails_batch  = ALL_EMAILS[:n]
        gt_batch      = ALL_GT[:n]

        print(f"LLM_TIER={tier}  |  batch={n}  |  "
              f"auto_mode={w_auto_mode.value}  |  "
              f"auto_approve={w_auto_rate.value:.0%}")
        run_button.disabled = True
        run_button.description = "⏳  Running…"

        try:
            RUNNER = HitlBatchRunner(
                emails=emails_batch,
                gt_classes=gt_batch,
                graph=GRAPH,
                auto_mode=w_auto_mode.value,
                auto_approve_rate=w_auto_rate.value,
            )
            RESULTS = RUNNER.run()

            # ── Save to runtime/benchmarks/latest.json (dashboard-compatible) ──
            # Field mapping: terminal_sink → predicted_sink, total_latency_ms → latency_ms,
            #                total_cost_usd → cost_usd, errors (list) → error (str|None)
            import datetime
            benchmarks_dir = REPO_ROOT / "runtime" / "benchmarks"
            benchmarks_dir.mkdir(parents=True, exist_ok=True)

            valid_r = [r for r in RESULTS if r.get("terminal_sink") is not None]
            per_record_out = [{
                "email_id":      r["email_id"],
                "gt_class":      r["gt_class"],
                "expected_sink": r["expected_sink"],
                "predicted_sink": r["terminal_sink"],
                "correct":       r["terminal_sink"] == r["expected_sink"],
                "latency_ms":    r.get("total_latency_ms", 0),
                "cost_usd":      r.get("total_cost_usd", 0.0),
                "error":         "; ".join(r.get("errors", [])) or None,
                # HITL fields — present in interactive runs; absent in old CLI eval exports
                "hitl_required":  r.get("hitl_required", False),
                "human_decision": r.get("human_decision"),
            } for r in RESULTS]

            pi_v  = [r for r in valid_r if r["expected_sink"] == "NewLead"]
            tp_   = sum(1 for r in pi_v   if r["terminal_sink"] == "NewLead")
            fp_   = sum(1 for r in valid_r if r["terminal_sink"] == "NewLead" and r["expected_sink"] != "NewLead")
            fn_   = sum(1 for r in pi_v   if r["terminal_sink"] != "NewLead")
            non_spam = [r for r in valid_r if r["gt_class"] != "spam"]
            spam_fp_ = sum(1 for r in non_spam if r["terminal_sink"] == "Refused-Spam")
            lats     = [r.get("total_latency_ms", 0) for r in RESULTS]
            costs    = [r.get("total_cost_usd", 0.0) for r in RESULTS]

            def _pct(lst, p):
                if not lst: return 0
                s = sorted(lst); k = (len(s) - 1) * p / 100
                lo, hi = int(k), min(int(k) + 1, len(s) - 1)
                return s[lo] + (s[hi] - s[lo]) * (k - lo)

            kpis_out = {
                "lead_precision":  tp_ / (tp_ + fp_)       if (tp_ + fp_)   else float("nan"),
                "lead_recall":     tp_ / (tp_ + fn_)       if (tp_ + fn_)   else float("nan"),
                "spam_fpr":        spam_fp_ / len(non_spam) if non_spam      else 0.0,
                "accuracy":        sum(1 for r in valid_r if r["terminal_sink"] == r["expected_sink"]) / len(valid_r) if valid_r else 0.0,
                "mean_latency_ms": sum(lats) / len(lats) if lats else 0,
                "p90_latency_ms":  _pct(lats, 90),
                "p95_latency_ms":  _pct(lats, 95),
                "total_cost_usd":  sum(costs),
                "n_records":       len(RESULTS),
                "n_errors":        len(RESULTS) - len(valid_r),
                "tp": tp_, "fp": fp_, "fn": fn_,
            }
            run_id_out = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ") + "-interactive"
            report_out = {
                "run_id":    run_id_out,
                "commit":    "interactive",
                "llm_tier":  tier,
                "kpis":      kpis_out,
                "per_record": per_record_out,
            }
            out_path = benchmarks_dir / "latest.json"
            out_path.write_text(json.dumps(report_out, indent=2, default=str))
            print(f"💾  Saved {len(RESULTS)} records → runtime/benchmarks/latest.json")
            print(f"    (re-run the dashboard notebook to pick up these results)")
            print(f"\n🎯  Pipeline complete — scroll down for results")
        except Exception:
            traceback.print_exc()
        finally:
            run_button.disabled = False
            run_button.description = "▶  Run Pipeline"

run_button.on_click(on_run_clicked)
display(run_button, run_output)

Button(button_style='primary', description='▶  Run Pipeline', icon='play', layout=Layout(height='44px', width=…

Output()

---
## Results
*Run the pipeline first (click **▶ Run Pipeline** above), then execute the cells below.*

In [8]:
"""Routing distribution — pie chart of terminal sinks."""
if not RESULTS:
    print("⚠️  No results yet — run the pipeline first.")
else:
    df = pd.DataFrame(RESULTS)
    sink_counts = df["terminal_sink"].value_counts().reset_index()
    sink_counts.columns = ["sink", "count"]

    fig = px.pie(
        sink_counts, names="sink", values="count",
        title="Email Routing Distribution",
        color_discrete_sequence=px.colors.qualitative.Set2,
    )
    fig.update_traces(textinfo="label+percent+value")
    fig.update_layout(legend_title="Sink", height=420)
    fig.show()

In [9]:
"""Confusion matrix — predicted sink vs expected sink."""
if not RESULTS:
    print("⚠️  No results yet.")
else:
    df = pd.DataFrame(RESULTS)

    # Drop rows where terminal_sink is None (pipeline error / unprocessed email).
    # Both the matrix AND the key metrics below use `valid` so counts are consistent.
    valid = df.dropna(subset=["terminal_sink", "expected_sink"]).copy()
    valid["correct"] = valid["terminal_sink"] == valid["expected_sink"]
    dropped = len(df) - len(valid)
    if dropped:
        error_ids = df[df["terminal_sink"].isna()]["email_id"].tolist()
        print(f"⚠️  {dropped} email(s) have no terminal_sink (pipeline error) — excluded from all stats.")
        print(f"   email_ids: {error_ids}\n")

    sinks_all = sorted(set(valid["expected_sink"]) | set(valid["terminal_sink"]))

    cm_data = pd.crosstab(
        valid["expected_sink"].rename("Expected"),
        valid["terminal_sink"].rename("Predicted"),
    ).reindex(index=sinks_all, columns=sinks_all, fill_value=0)

    fig = go.Figure(data=go.Heatmap(
        z=cm_data.values,
        x=cm_data.columns.tolist(),
        y=cm_data.index.tolist(),
        text=cm_data.values,
        texttemplate="%{text}",
        colorscale="Blues",
        showscale=True,
    ))
    fig.update_layout(
        title=f"Confusion Matrix (Expected vs Predicted Sink)  [{len(valid)}/{len(df)} emails]",
        xaxis_title="Predicted",
        yaxis_title="Expected",
        height=450,
        yaxis={"autorange": "reversed"},
    )
    fig.show()

    # Key metrics — computed on `valid` only, consistent with the matrix above
    acc = valid["correct"].mean()
    pi_mask = valid["expected_sink"] == "NewLead"
    if pi_mask.any():
        pi_precision_denom = (valid["terminal_sink"] == "NewLead").sum()
        pi_tp = ((valid["terminal_sink"] == "NewLead") & pi_mask).sum()
        pi_fn = ((valid["terminal_sink"] != "NewLead") & pi_mask).sum()
        precision = pi_tp / pi_precision_denom if pi_precision_denom else float("nan")
        recall    = pi_tp / (pi_tp + pi_fn) if (pi_tp + pi_fn) else float("nan")
        print(f"\n{'Overall accuracy':30s}: {acc:.1%}  (on {len(valid)}/{len(df)} emails)")
        print(f"{'PI Lead precision':30s}: {precision:.1%}  (TP={pi_tp}, FP={pi_precision_denom-pi_tp})")
        print(f"{'PI Lead recall':30s}: {recall:.1%}  (TP={pi_tp}, FN={pi_fn})")
    else:
        print(f"Overall accuracy: {acc:.1%}  (no PI leads in batch, {len(valid)}/{len(df)} emails)")

⚠️  8 email(s) have no terminal_sink (pipeline error) — excluded from all stats.
   email_ids: ['f143262f-dc5c-4eed-8da0-365bf89897b9', 'e0c53cb8-3da9-42a9-8ed4-2f1a3d4cbf37', '122c9a56-01d7-4256-b860-2ab696a402f2', 'd5704f32-702c-4d20-a862-18b848f4ef12', 'cc530e36-addc-4e13-ab3b-4d37560c95ee', 'd4f8fd72-f382-4cfd-8083-b73a473bd358', '5e6fea07-c453-4f1d-8199-2fdfb31022f0', 'bb2488a3-d363-47b6-af81-cf4f7701f7bb']




Overall accuracy              : 100.0%  (on 42/50 emails)
PI Lead precision             : 100.0%  (TP=18, FP=0)
PI Lead recall                : 100.0%  (TP=18, FN=0)


In [10]:
"""HITL decisions breakdown."""
if not RESULTS:
    print("⚠️  No results yet.")
else:
    df = pd.DataFrame(RESULTS)
    hitl_df = df[df["hitl_required"] == True].copy()

    print(f"Emails requiring HITL review: {len(hitl_df)} / {len(df)}")

    if hitl_df.empty:
        print("No HITL reviews triggered in this batch.")
    else:
        decision_counts = hitl_df["human_decision"].value_counts()
        fig = px.bar(
            decision_counts.reset_index(),
            x="human_decision", y="count",
            title="HITL Human Decisions",
            labels={"human_decision": "Decision", "count": "Count"},
            color="human_decision",
            color_discrete_map={
                "approve": "#4caf50",
                "reject": "#f44336",
                "reclassify": "#ff9800",
            },
        )
        fig.update_layout(showlegend=False, height=300)
        fig.show()

        # Score distribution for HITL emails
        valid_scores = hitl_df.dropna(subset=["appraisal_score"])
        if not valid_scores.empty:
            fig2 = px.histogram(
                valid_scores, x="appraisal_score", nbins=10,
                color="human_decision",
                title="Appraisal Score Distribution for HITL Emails",
                labels={"appraisal_score": "Appraisal Score", "count": "Count"},
                color_discrete_map={
                    "approve": "#4caf50", "reject": "#f44336", "reclassify": "#ff9800",
                },
            )
            fig2.update_layout(height=320, bargap=0.05)
            fig2.show()

Emails requiring HITL review: 18 / 50


In [11]:
"""Latency histogram + cost summary."""
if not RESULTS:
    print("⚠️  No results yet.")
else:
    df = pd.DataFrame(RESULTS)

    # Latency
    valid_lat = df[df["total_latency_ms"] > 0]
    if not valid_lat.empty:
        fig = px.histogram(
            valid_lat, x="total_latency_ms", nbins=20,
            title="End-to-End Latency Distribution (ms)",
            labels={"total_latency_ms": "Latency (ms)"},
            color_discrete_sequence=["#2196F3"],
        )
        fig.add_vline(
            x=30_000, line_dash="dash", line_color="red",
            annotation_text="30 s SLA", annotation_position="top right"
        )
        p95 = valid_lat["total_latency_ms"].quantile(0.95)
        fig.add_vline(
            x=p95, line_dash="dot", line_color="orange",
            annotation_text=f"p95={p95:.0f}ms"
        )
        fig.update_layout(height=320)
        fig.show()
    else:
        print("Latency data not available (tier3 stub run).")

    # Cost
    total_cost = df["total_cost_usd"].sum()
    mean_cost  = df["total_cost_usd"].mean()
    print(f"\nTotal cost : ${total_cost:.6f}")
    print(f"Mean/email : ${mean_cost:.6f}")
    print(f"Projected 1k emails: ${mean_cost * 1000:.4f}")


Total cost : $1.952858
Mean/email : $0.039057
Projected 1k emails: $39.0572


In [14]:
"""Executive summary card."""
if not RESULTS:
    print("⚠️  No results yet.")
else:
    df = pd.DataFrame(RESULTS)

    # Exclude records where the pipeline errored (terminal_sink=None).
    # All metrics must be computed on the same valid subset so they agree
    # with the confusion matrix above.
    valid = df.dropna(subset=["terminal_sink"]).copy()
    dropped = len(df) - len(valid)
    if dropped:
        print(f"⚠️  {dropped} email(s) excluded from stats (pipeline error — no terminal_sink).\n")

    total    = len(valid)           # emails included in stats
    total_in = len(df)              # emails attempted (shown separately)
    pi_mask  = valid["expected_sink"] == "NewLead"
    pi_total = pi_mask.sum()

    pi_tp = ((valid["terminal_sink"] == "NewLead") & pi_mask).sum()
    pi_fn = ((valid["terminal_sink"] != "NewLead") & pi_mask).sum()
    pi_fp = ((valid["terminal_sink"] == "NewLead") & ~pi_mask).sum()
    precision = pi_tp / (pi_tp + pi_fp) if (pi_tp + pi_fp) else float("nan")
    recall    = pi_tp / (pi_tp + pi_fn) if (pi_tp + pi_fn) else float("nan")

    accuracy  = (valid["terminal_sink"] == valid["expected_sink"]).mean()
    hitl_pct  = valid["hitl_required"].mean() if "hitl_required" in valid.columns else float("nan")
    total_cost= df["total_cost_usd"].sum()      # cost over all records, including errors
    p95_lat   = valid["total_latency_ms"].quantile(0.95)
    tier      = os.environ.get("LLM_TIER", "?")

    def _fmt(v, fmt):
        try:
            return format(v, fmt)
        except (ValueError, TypeError):
            return "N/A"

    def _kpi(label, value, target=None, good_low=False):
        color = "#333"
        if target is not None:
            try:
                v = float(str(value).replace("%","").replace("$","").replace("ms",""))
                t = float(target)
                color = "#388E3C" if (v >= t if not good_low else v <= t) else "#D32F2F"
            except Exception:
                pass
        return (
            f"<div style='text-align:center;padding:12px 16px'>"
            f"<div style='font-size:1.8em;font-weight:bold;color:{color}'>{value}</div>"
            f"<div style='font-size:0.8em;color:#666;margin-top:4px'>{label}</div>"
            f"</div>"
        )

    display(HTML(f"""
    <div style="border:1px solid #ddd;border-radius:10px;padding:20px;
                font-family:system-ui,sans-serif;max-width:900px;margin-top:16px">
      <h2 style="margin:0 0 16px;color:#1565C0">Executive Summary</h2>
      <div style="display:grid;grid-template-columns:repeat(4,1fr);
                  gap:8px;background:#f5f5f5;border-radius:8px;padding:8px">
        {_kpi("PI Lead Precision", _fmt(precision, '.1%'), target="80")}
        {_kpi("PI Lead Recall",    _fmt(recall,    '.1%'), target="80")}
        {_kpi("Overall Accuracy",  _fmt(accuracy,  '.1%'), target="75")}
        {_kpi("HITL Rate",         _fmt(hitl_pct,  '.1%'))}
      </div>
      <div style="display:grid;grid-template-columns:repeat(4,1fr);
                  gap:8px;margin-top:8px;background:#f5f5f5;border-radius:8px;padding:8px">
        {_kpi("Emails in stats", f"{total} / {total_in}")}
        {_kpi("PI Leads in batch",  pi_total)}
        {_kpi("p95 Latency",        _fmt(p95_lat, '.0f') + ' ms', target="30000", good_low=True)}
        {_kpi("Total LLM Cost",     '$' + _fmt(total_cost, '.5f'))}
      </div>
      <p style="margin:12px 0 0;font-size:0.82em;color:#888">
        Run on <b>{tier}</b> tier &nbsp;|&nbsp; ✅ = meets target &nbsp;|&nbsp;
        &#x274C; = below target &nbsp;|&nbsp; Targets: Precision &ge;80%, Recall &ge;80%, p95 &lt;30s
        {"&nbsp;|&nbsp; ⚠️ " + str(dropped) + " email(s) excluded (error)" if dropped else ""}
      </p>
    </div>
    """))

⚠️  8 email(s) excluded from stats (pipeline error — no terminal_sink).

